# Non-EEG personalized stress / anomaly model

Training notebook for the **MDM IoT Group 3** project.

- Dataset: [PhysioNet Non-EEG dataset v1.0.0](https://physionet.org/content/noneeg/1.0.0/)
- Inputs used: `ax`, `ay`, `az`, `temp` at 8 Hz and `SpO2`, `hr` at 1 Hz
- Excluded from deployment features: EDA (the final ESP32 node has no EDA sensor)
- Task: binary `normal` vs `stress`, plus personalized anomaly scoring
- Evaluation: leave-one-subject-out (LOSO) across 20 subjects
- Outputs: metrics, plots, and deployable JSON/TFLite/C-header artifacts

**Run this in Google Colab or Kaggle.** Do not download or train the dataset on the project PC.


## 1. Environment and paths

The first cell installs only `wfdb`. Colab and Kaggle already provide NumPy, pandas, scikit-learn, SciPy, matplotlib, and TensorFlow.


In [ ]:
%pip install -q wfdb


In [ ]:
from bisect import bisect_right
from pathlib import Path
import json
import math
import re
import urllib.request
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from scipy.stats import chi2

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

if Path("/content").exists():
    WORK = Path("/content")
elif Path("/kaggle/working").exists():
    WORK = Path("/kaggle/working")
else:
    raise RuntimeError("Run this notebook in Colab or Kaggle, not on the project PC.")

DATA_URL = "https://physionet.org/content/noneeg/get-zip/1.0.0/"
ZIP_PATH = WORK / "noneeg.zip"
DATA_DIR = WORK / "noneeg"
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("Working directory:", WORK)
print("Artifacts:", ARTIFACTS)


## 2. Download and locate the dataset

The ZIP layout is found dynamically with `rglob`, so this works whether the archive extracts into a subdirectory or directly into `noneeg/`.


In [ ]:
def find_dataset_root(root: Path) -> Path:
    matches = sorted(root.rglob("subjectinfo.csv"))
    if not matches:
        raise FileNotFoundError("subjectinfo.csv was not found after extraction")
    return matches[0].parent


if not list(DATA_DIR.rglob("Subject*_AccTempEDA.hea")):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading:", DATA_URL)
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Downloaded bytes:", ZIP_PATH.stat().st_size)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_DIR)

DATASET_ROOT = find_dataset_root(DATA_DIR)
recordings = sorted(
    DATASET_ROOT.glob("Subject*_AccTempEDA.hea"),
    key=lambda path: int(re.search(r"Subject(\d+)_", path.name).group(1)),
)

if len(recordings) != 20:
    raise RuntimeError(f"Expected 20 subjects, found {len(recordings)} in {DATASET_ROOT}")

subject_info = pd.read_csv(DATASET_ROOT / "subjectinfo.csv")
print("Dataset root:", DATASET_ROOT)
print("Recordings:", len(recordings))
print("Subjects in metadata:", len(subject_info))
subject_info.head()


## 3. Inspect the annotations

The `.atr` file is attached to the `AccTempEDA` record at 8 Hz. The observed labels are:

- `Relax` → normal
- `PhysicalStress`, `EmotionalStress`, `CognitiveStress` → stress

Some exports may expose generic 7/8-stage labels. The printout below is the source of truth. If any label is unmapped, fill `MANUAL_STAGE_MAP` in the next cell and rerun it.


In [ ]:
def clean_note(note) -> str:
    return str(note or "").replace("\x00", " ").strip()


sample_base = recordings[0].with_suffix("")
sample_ann = wfdb.rdann(str(sample_base), "atr")
annotation_preview = pd.DataFrame(
    {
        "sample": sample_ann.sample,
        "time_s": sample_ann.sample / 8.0,
        "symbol": sample_ann.symbol,
        "aux_note": [clean_note(note) for note in sample_ann.aux_note],
    }
)
print("Annotation count:", len(annotation_preview))
print(annotation_preview.to_string(index=False))


### Label mapping

Keyword mapping is used first. `MANUAL_STAGE_MAP` is intentionally editable for datasets whose annotations use short stage codes.


In [ ]:
NORMAL_KEYWORDS = ("relax", "rest", "baseline", "calm")
STRESS_KEYWORDS = (
    "stress",
    "mental",
    "physical",
    "emotional",
    "cognitive",
    "arithmetic",
    "stroop",
)

MANUAL_STAGE_MAP = {}


def stage_from_note(note) -> str | None:
    cleaned = clean_note(note)
    lowered = cleaned.lower().replace("_", " ").replace("-", " ")
    if not lowered:
        return None
    if cleaned in MANUAL_STAGE_MAP:
        return MANUAL_STAGE_MAP[cleaned]
    compact = lowered.replace(" ", "")
    if any(keyword in compact for keyword in NORMAL_KEYWORDS):
        return "normal"
    if any(keyword in compact for keyword in STRESS_KEYWORDS):
        return "stress"
    return None


unknown_notes = sorted(
    {
        clean_note(note)
        for note in sample_ann.aux_note
        if clean_note(note) and stage_from_note(note) is None
    }
)
print("Unmapped annotation labels:", unknown_notes)
if unknown_notes:
    print("Fill MANUAL_STAGE_MAP if necessary, for example {'A': 'normal', 'B': 'stress'}, then rerun this cell.")


## 4. Load signals and build 30 s windows

- Accelerometer/temperature stream: 8 Hz
- SpO2/heart-rate stream: 1 Hz, linearly interpolated onto the 8 Hz clock
- Window: 30 s, step: 15 s
- A window is kept only when at least 80% of its probe points share one annotation stage
- Features deliberately exclude EDA and do not claim true HRV; `hr_std` and `hr_slope` are HR-variability proxies


In [ ]:
FS_ACC = 8.0
FS_SPO2 = 1.0
WINDOW_S = 30
STEP_S = 15
MIN_CALIBRATION_S = 120

ALL_FEATURES = [
    "hr_mean",
    "hr_std",
    "hr_min",
    "hr_max",
    "hr_slope",
    "spo2_mean",
    "spo2_std",
    "spo2_min",
    "temp_mean",
    "temp_std",
    "temp_slope",
    "acc_mean",
    "acc_std",
    "acc_max",
    "acc_energy",
]

# ---- Choose what the deployed model uses -------------------------------------------
# True  : all 15 features (default, matches the committed artifacts)
# False : drop the 3 temperature features. Recommended if the ESP32's LM35 (finger skin
#         temperature) differs a lot from the dataset's wrist temperature. The server
#         adapts automatically to whatever feature list is in the exported JSON.
USE_TEMPERATURE = True
FEATURES = [name for name in ALL_FEATURES if USE_TEMPERATURE or not name.startswith("temp_")]
print("Training with", len(FEATURES), "features:", FEATURES)


def signal_index(record, name: str) -> int:
    for index, signal_name in enumerate(record.sig_name):
        if signal_name.strip().lower() == name.lower():
            return index
    raise KeyError(f"Signal {name!r} not found in {record.sig_name}")


def fill_missing(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if finite.all():
        return values
    if not finite.any():
        return np.zeros_like(values)
    indices = np.arange(len(values))
    return np.interp(indices, indices[finite], values[finite])


def annotation_events(annotation):
    events = []
    for sample, note in zip(annotation.sample, annotation.aux_note):
        stage = stage_from_note(note)
        if stage is not None:
            events.append((int(sample), stage))
    events.sort()
    if not events:
        raise RuntimeError("No mapped annotations found")
    return events


def load_subject(header_path: Path) -> dict:
    base = header_path.with_suffix("")
    acc_record = wfdb.rdrecord(str(base))
    spo2_base = base.with_name(base.name.replace("_AccTempEDA", "_SpO2HR"))
    spo2_record = wfdb.rdrecord(str(spo2_base))
    annotation = wfdb.rdann(str(base), "atr")

    ax = fill_missing(acc_record.p_signal[:, signal_index(acc_record, "ax")])
    ay = fill_missing(acc_record.p_signal[:, signal_index(acc_record, "ay")])
    az = fill_missing(acc_record.p_signal[:, signal_index(acc_record, "az")])
    temperature = fill_missing(acc_record.p_signal[:, signal_index(acc_record, "temp")])

    acc_time = np.arange(len(temperature)) / FS_ACC
    spo2_time = np.arange(spo2_record.p_signal.shape[0]) / FS_SPO2
    heart_rate = fill_missing(spo2_record.p_signal[:, signal_index(spo2_record, "hr")])
    oxygen = fill_missing(spo2_record.p_signal[:, signal_index(spo2_record, "SpO2")])

    heart_rate = np.interp(acc_time, spo2_time, heart_rate)
    oxygen = np.interp(acc_time, spo2_time, oxygen)

    events = annotation_events(annotation)
    event_starts = [event[0] for event in events]

    def stage_at(sample_index: int) -> str | None:
        position = bisect_right(event_starts, sample_index) - 1
        return events[position][1] if position >= 0 else None

    return {
        "subject": int(re.search(r"Subject(\d+)_", header_path.name).group(1)),
        "ax": ax,
        "ay": ay,
        "az": az,
        "temp": temperature,
        "hr": heart_rate,
        "spo2": oxygen,
        "stage_at": stage_at,
        "n_samples": len(temperature),
    }


def window_stage(stage_lookup, start: int, end: int) -> str | None:
    probes = np.linspace(start, end - 1, 12).astype(int)
    labels = [stage_lookup(int(index)) for index in probes]
    labels = [label for label in labels if label is not None]
    if not labels:
        return None
    values, counts = np.unique(labels, return_counts=True)
    if counts.max() / len(probes) < 0.8:
        return None
    return str(values[counts.argmax()])


def safe_stat(values, function) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(function(values)) if values.size else float("nan")


def slope(values, hz: float = FS_ACC) -> float:
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if finite.sum() < 2:
        return float("nan")
    x = np.arange(len(values))[finite] / hz
    return float(np.polyfit(x, values[finite], 1)[0])


def extract_all_features(record: dict, start: int, end: int) -> list[float]:
    hr_window = record["hr"][start:end]
    spo2_window = record["spo2"][start:end]
    temp_window = record["temp"][start:end]
    magnitude = np.sqrt(record["ax"][start:end] ** 2 + record["ay"][start:end] ** 2 + record["az"][start:end] ** 2)
    return [
        safe_stat(hr_window, np.mean),
        safe_stat(hr_window, np.std),
        safe_stat(hr_window, np.min),
        safe_stat(hr_window, np.max),
        slope(hr_window, FS_ACC),
        safe_stat(spo2_window, np.mean),
        safe_stat(spo2_window, np.std),
        safe_stat(spo2_window, np.min),
        safe_stat(temp_window, np.mean),
        safe_stat(temp_window, np.std),
        slope(temp_window, FS_ACC),
        safe_stat(magnitude, np.mean),
        safe_stat(magnitude, np.std),
        safe_stat(magnitude, np.max),
        safe_stat(magnitude, lambda values: np.mean(values**2)),
    ]


def extract_features(record: dict, start: int, end: int) -> list[float]:
    by_name = dict(zip(ALL_FEATURES, extract_all_features(record, start, end)))
    return [by_name[name] for name in FEATURES]


window_samples = int(WINDOW_S * FS_ACC)
step_samples = int(STEP_S * FS_ACC)
rows = []

for header_path in recordings:
    record = load_subject(header_path)
    for start in range(0, record["n_samples"] - window_samples + 1, step_samples):
        end = start + window_samples
        stage = window_stage(record["stage_at"], start, end)
        if stage is None:
            continue
        values = extract_features(record, start, end)
        if not np.isfinite(values).all():
            continue
        row = {
            "subject": record["subject"],
            "window_start_s": start / FS_ACC,
            "window_end_s": end / FS_ACC,
            "stage": stage,
        }
        row.update(dict(zip(FEATURES, values)))
        rows.append(row)

windows = pd.DataFrame(rows)
windows["label"] = (windows["stage"] == "stress").astype(int)
if windows.empty:
    raise RuntimeError("No windows were produced; check the annotation mapping")

print("Window count:", len(windows))
print("Subjects:", windows["subject"].nunique())
print(windows.groupby(["subject", "stage"]).size().unstack(fill_value=0).to_string())
windows.to_csv(ARTIFACTS / "windows.csv", index=False)
windows.head()


## 5. Leave-one-subject-out evaluation

Every test subject is completely held out. The supervised models are trained on the other 19 subjects. Personalization baselines calibrate on only the first 120 s of normal windows from the held-out subject, then score later windows.

`random_forest` is included as an upper-bound reference; it is not the intended ESP32 model.


In [ ]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


def supervised_models() -> dict:
    return {
        "logreg": Pipeline(
            [
                ("scale", StandardScaler()),
                ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)),
            ]
        ),
        "linear_svm": Pipeline(
            [
                ("scale", StandardScaler()),
                ("model", LinearSVC(class_weight="balanced", random_state=42)),
            ]
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42,
        ),
    }


def decision_score(model, values) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(values)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(values)
    return model.predict(values).astype(float)


def metrics_for(y_true, y_pred, score=None) -> dict:
    result = {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "n_windows": int(len(y_true)),
        "n_stress": int(np.sum(y_true == 1)),
    }
    if score is not None and len(np.unique(y_true)) > 1:
        result["roc_auc"] = roc_auc_score(y_true, score)
    else:
        result["roc_auc"] = float("nan")
    return result


def calibration_stats(frame: pd.DataFrame):
    calibration = frame[(frame["window_start_s"] < MIN_CALIBRATION_S) & (frame["label"] == 0)]
    if len(calibration) < 4:
        return None
    values = calibration[FEATURES].to_numpy()
    center = np.median(values, axis=0)
    mad = np.median(np.abs(values - center), axis=0) * 1.4826
    scale = np.where(mad > 1e-6, mad, 1.0)
    return center, scale


def normalized_matrix(frame: pd.DataFrame):
    xs, ys = [], []
    for _, subject_frame in frame.groupby("subject"):
        stats = calibration_stats(subject_frame)
        if stats is None:
            continue
        values = subject_frame[FEATURES].to_numpy()
        xs.append((values - stats[0]) / stats[1])
        ys.append(subject_frame["label"].to_numpy())
    if not xs:
        return np.empty((0, len(FEATURES))), np.empty(0)
    return np.vstack(xs), np.concatenate(ys)


subjects = sorted(windows["subject"].unique())
fold_rows = []
prediction_frames = []

for held_out in subjects:
    train = windows[windows["subject"] != held_out]
    test = windows[windows["subject"] == held_out]
    X_train = train[FEATURES].to_numpy()
    y_train = train["label"].to_numpy()
    X_test = test[FEATURES].to_numpy()
    y_test = test["label"].to_numpy()

    for name, model in supervised_models().items():
        model.fit(X_train, y_train)
        score = decision_score(model, X_test)
        threshold = 0.5 if name in {"logreg", "random_forest"} else 0.0
        prediction = (score >= threshold).astype(int)
        fold_rows.append({"model": name, "subject": held_out, **metrics_for(y_test, prediction, score)})
        prediction_frames.append(
            pd.DataFrame(
                {
                    "model": name,
                    "subject": held_out,
                    "y_true": y_test,
                    "y_pred": prediction,
                    "score": score,
                }
            )
        )

    calibrated_stats = calibration_stats(test)
    if calibrated_stats is not None:
        X_train_personal, y_train_personal = normalized_matrix(train)
        if len(X_train_personal) > 0:
            personal_lr = LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42,
            )
            personal_lr.fit(X_train_personal, y_train_personal)
            evaluation = test[test["window_start_s"] >= MIN_CALIBRATION_S]
            if len(evaluation) > 0:
                X_evaluation = (evaluation[FEATURES].to_numpy() - calibrated_stats[0]) / calibrated_stats[1]
                y_evaluation = evaluation["label"].to_numpy()
                score = personal_lr.predict_proba(X_evaluation)[:, 1]
                prediction = (score >= 0.5).astype(int)
                fold_rows.append(
                    {
                        "model": "personal_logreg",
                        "subject": held_out,
                        **metrics_for(y_evaluation, prediction, score),
                    }
                )
                prediction_frames.append(
                    pd.DataFrame(
                        {
                            "model": "personal_logreg",
                            "subject": held_out,
                            "y_true": y_evaluation,
                            "y_pred": prediction,
                            "score": score,
                        }
                    )
                )

    normal_train = train[train["label"] == 0]
    isolation = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=42,
        n_jobs=-1,
    )
    isolation.fit(normal_train[FEATURES].to_numpy())
    isolation_score = -isolation.score_samples(X_test)
    isolation_prediction = (isolation.predict(X_test) == -1).astype(int)
    fold_rows.append(
        {
            "model": "isolation_forest",
            "subject": held_out,
            **metrics_for(y_test, isolation_prediction, isolation_score),
        }
    )
    prediction_frames.append(
        pd.DataFrame(
            {
                "model": "isolation_forest",
                "subject": held_out,
                "y_true": y_test,
                "y_pred": isolation_prediction,
                "score": isolation_score,
            }
        )
    )

    calibration = test[(test["window_start_s"] < MIN_CALIBRATION_S) & (test["label"] == 0)]
    evaluation = test[test["window_start_s"] >= MIN_CALIBRATION_S]
    if len(calibration) >= 4 and len(evaluation) > 0:
        X_calibration = calibration[FEATURES].to_numpy()
        X_evaluation = evaluation[FEATURES].to_numpy()
        y_evaluation = evaluation["label"].to_numpy()

        center = np.median(X_calibration, axis=0)
        mad = np.median(np.abs(X_calibration - center), axis=0) * 1.4826
        scale = np.where(mad > 1e-6, mad, 1.0)
        robust_z = np.max(np.abs((X_evaluation - center) / scale), axis=1)
        robust_prediction = (robust_z > 3.5).astype(int)
        fold_rows.append(
            {
                "model": "personal_robust_z",
                "subject": held_out,
                **metrics_for(y_evaluation, robust_prediction, robust_z),
            }
        )
        prediction_frames.append(
            pd.DataFrame(
                {
                    "model": "personal_robust_z",
                    "subject": held_out,
                    "y_true": y_evaluation,
                    "y_pred": robust_prediction,
                    "score": robust_z,
                }
            )
        )

        covariance = np.cov(X_calibration, rowvar=False)
        covariance = covariance + np.eye(len(FEATURES)) * max(1e-8, np.trace(covariance) / len(FEATURES) * 1e-6)
        inverse_covariance = np.linalg.pinv(covariance)
        differences = X_evaluation - center
        mahalanobis = np.sqrt(
            np.einsum("ij,jk,ik->i", differences, inverse_covariance, differences)
        )
        mahalanobis_threshold = math.sqrt(chi2.ppf(0.99, df=len(FEATURES)))
        mahalanobis_prediction = (mahalanobis > mahalanobis_threshold).astype(int)
        fold_rows.append(
            {
                "model": "personal_mahalanobis",
                "subject": held_out,
                **metrics_for(y_evaluation, mahalanobis_prediction, mahalanobis),
            }
        )
        prediction_frames.append(
            pd.DataFrame(
                {
                    "model": "personal_mahalanobis",
                    "subject": held_out,
                    "y_true": y_evaluation,
                    "y_pred": mahalanobis_prediction,
                    "score": mahalanobis,
                }
            )
        )

folds = pd.DataFrame(fold_rows)
summary = (
    folds.groupby("model")
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        f1_mean=("f1", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        folds=("subject", "nunique"),
    )
    .sort_values("balanced_accuracy_mean", ascending=False)
)

folds.to_csv(ARTIFACTS / "loso_folds.csv", index=False)
summary.to_csv(ARTIFACTS / "loso_summary.csv")
predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_csv(ARTIFACTS / "loso_predictions.csv", index=False)

print(summary.round(3).to_string())


## 6. Metrics plots

The confusion matrix is for the best mean balanced accuracy. These are LOSO results, not training-set results.


In [ ]:
best_model = summary.index[0]
best_predictions = predictions[predictions["model"] == best_model]

matrix = confusion_matrix(best_predictions["y_true"], best_predictions["y_pred"], labels=[0, 1])
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].barh(
    summary.index,
    summary["balanced_accuracy_mean"],
    xerr=summary["balanced_accuracy_std"].fillna(0),
    color="#2a6f97",
)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("LOSO balanced accuracy")
axes[0].set_title("Model comparison")
axes[0].grid(axis="x", alpha=0.25)

image = axes[1].imshow(matrix, cmap="Blues")
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        axes[1].text(column, row, matrix[row, column], ha="center", va="center", fontsize=13)
axes[1].set_xticks([0, 1], ["normal", "stress"])
axes[1].set_yticks([0, 1], ["normal", "stress"])
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_title(f"Confusion matrix: {best_model}")

figure.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04)
figure.tight_layout()
figure.savefig(ARTIFACTS / "loso_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

print("Best LOSO model:", best_model)


## 7. Refit deployment artifacts

These models are refit on all subjects for deployment. Use `loso_summary.csv` for honest generalization claims; do not report the refit training score as LOSO performance.

Artifacts:

- `model_personal_logreg.json` — calibration-normalized logistic regression for server/edge use
- `model_lr.json` — global standardized logistic regression reference
- `model_robust_z.json` — diagonal personalized baseline for a small MCU
- `model_mahalanobis_server.json` — full-covariance personalized baseline for the server
- `model_mlp.tflite`, `model_mlp.h`, `model_mlp_meta.json` — tiny MLP for TFLite Micro


In [ ]:
X_all = windows[FEATURES].to_numpy()
y_all = windows["label"].to_numpy()

logreg = Pipeline(
    [
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42)),
    ]
)
logreg.fit(X_all, y_all)
scaler = logreg.named_steps["scale"]
classifier = logreg.named_steps["model"]

logreg_artifact = {
    "model_type": "logistic_regression",
    "features": FEATURES,
    "window_seconds": WINDOW_S,
    "step_seconds": STEP_S,
    "input_normalization": {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
    },
    "coefficients": classifier.coef_[0].tolist(),
    "intercept": float(classifier.intercept_[0]),
    "decision_threshold": 0.5,
    "note": "Deployment refit on all subjects; use LOSO metrics for generalization.",
}
(ARTIFACTS / "model_lr.json").write_text(json.dumps(logreg_artifact, indent=2))

X_personal, y_personal = normalized_matrix(windows)
personal_logreg = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42,
)
personal_logreg.fit(X_personal, y_personal)
personal_artifact = {
    "model_type": "personal_logreg",
    "features": FEATURES,
    "window_seconds": WINDOW_S,
    "step_seconds": STEP_S,
    "calibration_seconds": MIN_CALIBRATION_S,
    "normalization": {
        "method": "median_mad",
        "mad_scale": 1.4826,
        "min_scale": 1e-6,
    },
    "coefficients": personal_logreg.coef_[0].tolist(),
    "intercept": float(personal_logreg.intercept_[0]),
    "decision_threshold": 0.5,
    "note": "Server must compute center/scale from the first 120 s of normal feature windows per device, then z-score before scoring.",
}
(ARTIFACTS / "model_personal_logreg.json").write_text(
    json.dumps(personal_artifact, indent=2)
)

normal_windows = X_all[y_all == 0]
center = np.median(normal_windows, axis=0)
mad = np.median(np.abs(normal_windows - center), axis=0) * 1.4826
scale = np.where(mad > 1e-6, mad, 1.0)

robust_artifact = {
    "model_type": "personal_robust_z",
    "features": FEATURES,
    "window_seconds": WINDOW_S,
    "step_seconds": STEP_S,
    "center": center.tolist(),
    "scale": scale.tolist(),
    "threshold": 3.5,
    "note": "Recalibrate center/scale on the first 120 s of normal data per device.",
}
(ARTIFACTS / "model_robust_z.json").write_text(json.dumps(robust_artifact, indent=2))

mean = normal_windows.mean(axis=0)
covariance = np.cov(normal_windows, rowvar=False)
covariance = covariance + np.eye(len(FEATURES)) * max(1e-8, np.trace(covariance) / len(FEATURES) * 1e-6)
inverse_covariance = np.linalg.pinv(covariance)

mahalanobis_artifact = {
    "model_type": "personal_mahalanobis",
    "features": FEATURES,
    "window_seconds": WINDOW_S,
    "step_seconds": STEP_S,
    "mean": mean.tolist(),
    "inverse_covariance": inverse_covariance.tolist(),
    "threshold": float(math.sqrt(chi2.ppf(0.99, df=len(FEATURES)))),
    "note": "Global fallback; prefer center/covariance recalibrated per device.",
}
(ARTIFACTS / "model_mahalanobis_server.json").write_text(
    json.dumps(mahalanobis_artifact, indent=2)
)

deployment_contract = {
    "features": FEATURES,
    "window_seconds": WINDOW_S,
    "step_seconds": STEP_S,
    "sampling": {
        "acc_temp_hz": FS_ACC,
        "spo2_hr_hz": FS_SPO2,
    },
    "labels": {"0": "normal", "1": "stress"},
    "excluded_signals": ["EDA"],
    "calibration_seconds": MIN_CALIBRATION_S,
}
(ARTIFACTS / "deployment_contract.json").write_text(
    json.dumps(deployment_contract, indent=2)
)

print("Wrote:", sorted(path.name for path in ARTIFACTS.glob("*.json")))


In [ ]:
tflite_path = ARTIFACTS / "model_mlp.tflite"
header_path = ARTIFACTS / "model_mlp.h"
mlp_meta_path = ARTIFACTS / "model_mlp_meta.json"

try:
    import tensorflow as tf

    tf.keras.utils.set_random_seed(42)
    mlp_scaler = StandardScaler()
    X_scaled = mlp_scaler.fit_transform(X_all).astype(np.float32)
    class_weight = {
        0: 1.0,
        1: float((y_all == 0).sum() / max(1, (y_all == 1).sum())),
    }

    mlp = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(len(FEATURES),)),
            tf.keras.layers.Dense(24, activation="relu"),
            tf.keras.layers.Dense(12, activation="relu"),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )
    mlp.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    mlp.fit(
        X_scaled,
        y_all.astype(np.float32),
        epochs=80,
        batch_size=32,
        verbose=0,
        class_weight=class_weight,
    )

    converter = tf.lite.TFLiteConverter.from_keras_model(mlp)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    tflite_path.write_bytes(tflite_model)

    def c_array(data: bytes, name: str) -> str:
        lines = [f"alignas(8) const unsigned char {name}[] = {{"]
        for offset in range(0, len(data), 12):
            chunk = ", ".join(f"0x{byte:02x}" for byte in data[offset : offset + 12])
            lines.append(f"  {chunk},")
        lines.append("};")
        lines.append(f"const unsigned int {name}_len = {len(data)};")
        return "\n".join(lines) + "\n"

    header_path.write_text(c_array(tflite_model, "model_mlp_tflite"))
    mlp_meta = {
        "model_type": "mlp_24_12",
        "features": FEATURES,
        "window_seconds": WINDOW_S,
        "step_seconds": STEP_S,
        "input_normalization": {
            "mean": mlp_scaler.mean_.tolist(),
            "scale": mlp_scaler.scale_.tolist(),
        },
        "output": "sigmoid stress probability",
        "decision_threshold": 0.5,
        "tflite_bytes": len(tflite_model),
    }
    mlp_meta_path.write_text(json.dumps(mlp_meta, indent=2))
    print("TFLite bytes:", len(tflite_model))

except Exception as exc:
    print("TFLite export skipped or failed:", repr(exc))
    print("The JSON models are still valid; rerun this cell after checking TensorFlow.")


## 8. Package artifacts and record what to use where

- Server: load `model_lr.json` for supervised scoring, or `model_mahalanobis_server.json` after per-device calibration.
- ESP32: use `model_mlp.tflite` / `model_mlp.h` for TFLite Micro; apply the scaler from `model_mlp_meta.json` first.
- Always keep `deployment_contract.json` next to the server code so feature order and window sizes cannot drift.
- The first 120 s of normal data are for personal calibration; do not score them as anomalies.


In [ ]:
artifact_zip = WORK / "noneeg_artifacts.zip"
with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACTS.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print("Artifact package:", artifact_zip)
print("Size (bytes):", artifact_zip.stat().st_size)
for path in sorted(ARTIFACTS.iterdir()):
    if path.is_file():
        print(f"{path.name:36s} {path.stat().st_size:>9d} bytes")

print("\nDownload this file to the project, then integrate the model into server/anomaly.py:")
print(artifact_zip)
